In [34]:
import pandas as pd

pd.set_option('display.max_columns', None)

df = pd.read_parquet("data/games_2024_2025_2026.parquet")

# todays_games = df[df["game_date"] == "2026-04-08"]

home_favored = df[
    (df["sp_kbb_diff"] > 3) &
    (df["sp_xfip_diff"] < -0.5) &
    (df["offense_diff"] > 10)
]

away_favored = df[
    (df["sp_kbb_diff"] < -3) &
    (df["sp_xfip_diff"] > 0.5) &
    (df["offense_diff"] < -10)
]

mismatches = pd.concat([home_favored, away_favored])
# mismatches["favored_team"] = mismatches.apply(
#     lambda row: row["home_team_name"] if row["sp_kbb_diff"] > 0 else row["away_team_name"],
#     axis=1
# )

mismatches["favored_home"] = mismatches["sp_kbb_diff"] > 0
mismatches["favored_won"] = (
    (mismatches["favored_home"] & (mismatches["home_win"] == 1)) |
    (~mismatches["favored_home"] & (mismatches["home_win"] == 0))
)

mismatches["game_date"] = pd.to_datetime(mismatches["game_date"])

mismatches["year"] = mismatches["game_date"].dt.year
mismatches["month"] = mismatches["game_date"].dt.month

mismatches.groupby(["year", "month"]).agg(
    games=("favored_won", "size"),
    win_rate=("favored_won", "mean")
).sort_index()

games  win_rate
year month                 
2024 3         23  0.608696
     4         26  0.576923
     5         22  0.818182
     6         28  0.642857
     7         14  0.785714
     8         27  0.851852
     9         20  0.850000
2025 3         16  0.812500
     4         10  0.700000
     5         22  0.863636
     6         28  0.678571
     7          6  0.666667
     8         27  0.592593
     9         12  0.833333
2026 3         28  0.642857
     4         21  0.476190

In [20]:
import pandas as pd

pd.set_option('display.max_columns', None)

# Load
b24 = pd.read_parquet("data/stats_tables/team_batting_2024.parquet")
b25 = pd.read_parquet("data/stats_tables/team_batting_2025.parquet")
b26 = pd.read_parquet("data/stats_tables/team_batting_2026.parquet")

key = ["Team"]

def suffix_cols(df, year):
    return df.rename(columns={
        col: f"{col}_{year}" for col in df.columns if col not in key
    })

b24 = suffix_cols(b24, 2024)
b25 = suffix_cols(b25, 2025)
b26 = suffix_cols(b26, 2026)

# Merge
batting = b24.merge(b25, on=key, how="inner") \
             .merge(b26, on=key, how="inner")

# --- wRC+ YoY delta (2026 vs 2025) ---
batting["wRC+_delta_26_vs_25"] = batting["wRC+_2026"] - batting["wRC+_2025"]

# Sort descending (biggest improvement first)
batting = batting.sort_values(by="wRC+_delta_26_vs_25", ascending=False)
batting_filtered = batting[batting["Team"].isin(["KC","CHW"])]

batting_filtered.head(30)

,Team,wRC+_2024,wRC+_2025,wRC+_2026,wRC+_delta_26_vs_25
26,KC,99.742087,98.260264,99.303136,1.042871
12,CHW,86.940211,93.945720,90.011614,-3.934106


In [18]:
import pandas as pd

pd.set_option('display.max_columns', None)

# Load
p24 = pd.read_parquet("data/stats_tables/pitchers_2024.parquet")
p25 = pd.read_parquet("data/stats_tables/pitchers_2025.parquet")
p26 = pd.read_parquet("data/stats_tables/pitchers_2026.parquet")



key = ["Name"]

def suffix_cols(df, year):
    return df.rename(columns={
        col: f"{col}_{year}" for col in df.columns if col not in key
    })

p24 = suffix_cols(p24, 2024)
p25 = suffix_cols(p25, 2025)
p26 = suffix_cols(p26, 2026)

# Merge
pitching = p24.merge(p25, on=key, how="outer") \
             .merge(p26, on=key, how="outer")

# --- wRC+ YoY delta (2026 vs 2025) ---
# batting["wRC+_delta_26_vs_25"] = batting["wRC+_2026"] - batting["wRC+_2025"]

# # Sort descending (biggest improvement first)
# batting = batting.sort_values(by="wRC+_delta_26_vs_25", ascending=False)

pitching_filtered = pitching[pitching["Name"].isin(["Seth Lugo", "Anthony Kay"])]
pitching_filtered.head()

,Name,K%_2024,BB%_2024,xFIP_2024,kbb_2024,K%_2025,BB%_2025,xFIP_2025,kbb_2025,K%_2026,BB%_2026,xFIP_2026,kbb_2026
66,Anthony Kay,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12.500000,15.000000,6.877778,-2.500000
972,Seth Lugo,21.650718,5.741627,3.182258,15.909091,20.458265,9.001637,5.054128,11.456628,22.222222,4.444444,1.864706,17.777778
